# Exploring the lakehouse with SQL

The other notebook reads the Delta tables through `lakehouse.py`, by path. This
one gives them **names**, so the same data answers to
`spark.sql("SELECT * FROM gym_tracker.workout_logs_typed")`.

That needs two things beyond the usual pipeline:

1. The **Pipeline Metastore** add-on running.
2. The **Pipeline Spark** add-on's `metastore_uris` set to
   `thrift://172.30.32.1:9083`, and restarted.

Without them every cell below stops early and says so — nothing here breaks a
pipeline that has no metastore, and the path-based notebook keeps working either
way.

> **First run is slow.** Spark's built-in Hive client is 2.3.10 and cannot talk
> to a 4.1.0 metastore, so it downloads a matching client on the first query
> that touches the catalog. That is a one-time cost, cached under `/data/spark`.


In [ ]:
import os, sys
sys.path.insert(0, "/share/pipeline-airflow/lib")

from pyspark.sql import SparkSession, functions as F
from lakehouse import catalog_available, register, table, SCHEMAS

spark = SparkSession.builder.remote(
    os.environ.get("SPARK_CONNECT_URL", "sc://172.30.32.1:15002")
).getOrCreate()

HAVE_CATALOG = catalog_available(spark)
print("Hive catalog:", "yes" if HAVE_CATALOG else "no — set metastore_uris on Pipeline Spark")


## 1. Register the tables

`register()` writes names into the metastore. It is metadata only: nothing is
moved, copied or rewritten, and Delta keeps the schema in its own transaction
log, so a tracker gaining a column needs no re-run. Running it again is free.

Each table gets **two** names:

| Name | What it is |
|---|---|
| `workout_logs` | exactly as the merge wrote it — the payload is still a JSON string |
| `workout_logs_typed` | a view applying the schema, dropping deleted rows, with the change metadata as `_seq`, `_changed_at`, `_actor`, `_deleted_at` |

The typed view is almost always the one you want. The raw table is there for
when a tracker has added a column that `lakehouse.SCHEMAS` does not know about
yet — it is all still in the JSON.


In [ ]:
if HAVE_CATALOG:
    result = register(spark)
    print("registered:", len(result["registered"]))
    for name in result["registered"]:
        print("  ", name)
    if result["skipped"]:
        # Not an error: a table the DAGs have not written yet simply is not there.
        print("skipped (not in the lakehouse yet):", ", ".join(result["skipped"]))
else:
    print("no catalog — skipping")


## 2. What is in the catalog

In [ ]:
if HAVE_CATALOG:
    spark.sql("SHOW DATABASES").show()
    for db in ("gym_tracker", "coop_tracker"):
        try:
            print(f"--- {db} ---")
            spark.sql(f"SHOW TABLES IN {db}").show(truncate=False)
        except Exception as exc:
            print(f"  {db}: {exc}")


Where a table actually lives, and what Spark thinks its columns are. `Location`
is the `s3a://` path the DAGs write — registering did not move it.


In [ ]:
if HAVE_CATALOG:
    spark.sql("DESCRIBE FORMATTED gym_tracker.workout_logs").show(60, truncate=False)


## 3. The point

`SELECT *`, by name, no path anywhere.


In [ ]:
if HAVE_CATALOG:
    spark.sql("""
        SELECT ts, exercise_id, sets, reps, weight_kg, _actor
        FROM gym_tracker.workout_logs_typed
        ORDER BY ts DESC
        LIMIT 10
    """).show(truncate=False)


## 4. Two things that will mislead you

Same traps as the path-based notebook — the catalog does not remove them.

**`sets` is often null.** The app stores nothing for a single-set entry, so
`sets * reps` nulls those rows and whole days vanish from a total. Use
`COALESCE(sets, 1)`, which is what the app's own figures do and what
`lakehouse.total_reps()` does in Python.

**`_actor` is null for everything the bootstrap loaded.** A snapshot is state,
not a change, so the first load has no actor. It only becomes meaningful for
rows written after the trackers started feeding the pipeline.


In [ ]:
if HAVE_CATALOG:
    spark.sql("""
        SELECT substr(ts, 1, 10)              AS day,
               sum(coalesce(sets, 1) * reps)  AS reps,
               count(*)                       AS entries
        FROM gym_tracker.workout_logs_typed
        GROUP BY 1
        ORDER BY 1 DESC
        LIMIT 14
    """).show()


A plank is logged as duration rather than reps, so it contributes zero to the
figure above and would disappear from any volume total. `held_seconds` is the
same idea for time-based exercises.


In [ ]:
if HAVE_CATALOG:
    spark.sql("""
        SELECT e.name,
               count(*)                                    AS sessions,
               sum(coalesce(w.sets, 1) * w.reps)           AS reps,
               sum(coalesce(w.sets, 1) * w.duration_sec)   AS held_seconds
        FROM gym_tracker.workout_logs_typed w
        JOIN gym_tracker.exercises_typed e ON e.id = w.exercise_id
        GROUP BY e.name
        ORDER BY reps DESC NULLS LAST
    """).show(truncate=False)


## 5. Deleted rows, and the raw table

The typed view shows live rows only. Un-ticking a challenge really does delete
the workout it logged, and that is often the interesting question — so ask the
raw table, which keeps everything.


In [ ]:
if HAVE_CATALOG:
    spark.sql("""
        SELECT seq, actor, changed_at, deleted_at
        FROM gym_tracker.workout_logs
        WHERE deleted_at IS NOT NULL
        ORDER BY seq DESC
        LIMIT 10
    """).show(truncate=False)


And when a tracker has added a column that `SCHEMAS` does not know about yet,
it is still in the JSON payload — no re-registration, no DAG change:

```sql
SELECT get_json_object(data, '$.some_new_column') FROM gym_tracker.workout_logs
```


In [ ]:
if HAVE_CATALOG:
    spark.sql("""
        SELECT get_json_object(data, '$.notes') AS notes, seq
        FROM gym_tracker.workout_logs
        WHERE get_json_object(data, '$.notes') IS NOT NULL
        ORDER BY seq DESC
        LIMIT 5
    """).show(truncate=False)


## 6. Mixing SQL and the DataFrame API

A registered table is an ordinary Spark table, so `spark.table()` gives you a
DataFrame and everything in the other notebook still applies.


In [ ]:
if HAVE_CATALOG:
    df = spark.table("gym_tracker.workout_logs_typed")
    print(type(df).__name__, "with", len(df.columns), "columns")
    df.groupBy("_actor").count().show()


---

### Notes

- **Dropping a registered table does not delete data.** `DROP TABLE` removes the
  name; the Delta files stay where the DAGs wrote them, and `register()` brings
  the name back.
- **Losing the metastore loses names, not data.** Its catalog is a normal
  Postgres database; if it is lost, re-run `register()`.
- The path-based reader is unaffected by any of this — `lakehouse.table()` and
  `tables()` work with or without a catalog.


In [ ]:
from lakehouse import tables
gym = tables(spark, "gym_tracker")     # still works, catalog or not
sorted(gym)
